In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

In [3]:
from huggingface_hub import login
login()

In [4]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")

In [7]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
0


In [ ]:
data = load_dataset("HuggingFaceFW/fineweb-edu",
                       name="sample-10BT",
                       split="train",
                       streaming=False)



Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [ ]:




for exaple in data:
    
    
    print(exaple["text"][:100])
    
        
    break




The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are rea


0.0

In [6]:
def get_micro_batch():
    
    for text in data.take(1):
        text = text["text"]
        tokens = tokenizer.encode(text)
    tokens = torch.tensor(tokens[:100])
    x = tokens[:-1]
    y = tokens[1:]

    return x.unsqueeze(0) , y.unsqueeze(0)
    
x , y = get_micro_batch()

x,y 

(tensor([[  504, 18994, 12405,   198,  2193,   511,   260,  2606,    28, 18233,
            284, 26467,   281, 12405, 33015,   417,    99,  2905,    28,   732,
            502,   359,  2159,   563,   314,  4766,   284,  6967,    30, 14188,
            282,  2275,   284,   260,  4766,   288,  3525,    30,   198, 40168,
            417,    99, 25827,   282,  4994,    30, 18976,  2626,   282,  6724,
           4342,   354,  6967, 21134,  2269,   281,  5913,   884,   282,   260,
           1194,    30,  3488, 25827,   282,  4994,    30,  9117,   873,   979,
          14734,   411,  9846,  4342,   253,  1106,   282,  6967,   338,  2049,
           1272, 25174,   284, 45358,    30,   198,   504,  4766,  1041, 17223,
            281,  5087, 14750,  1272,   281,  1454, 40001,   282, 14755]]),
 tensor([[18994, 12405,   198,  2193,   511,   260,  2606,    28, 18233,   284,
          26467,   281, 12405, 33015,   417,    99,  2905,    28,   732,   502,
            359,  2159,   563,   314,  4766,

In [9]:
torch.save((x,y), "Debug_batch.pt")

In [ ]:
import os
import numpy as np
from tqdm.notebook import tqdm
import multiprocessing as mp

nprocs = max(1, os.cpu_count()//2)
SHARD_SIZE = 100_000_000
OUT_DIR = "tensor_data"

data = load_dataset("HuggingFaceFW/fineweb-edu",
                       name="sample-10BT",
                       split="train[:500]")


current_tokens = []

os.makedirs("tensor_data", exist_ok=True)

def tokenize(example):
    tokens = tokenizer.encode(example["text"])
    tokens_np = np.array(tokens , dtype=np.uint16)

    return tokens_np

def write_shard(args):
    idx, tokens = args
    path = os.path.join(OUT_DIR, f"shard_{idx:04d}.npy")
    np.save(path, tokens)
    print(f"Saved shard {idx} ({len(tokens):,} tokens)")

current_tokens = np.empty((SHARD_SIZE,), dtype=np.uint16)
token_count = 0
shard_idx = 0
progress = tqdm(total=SHARD_SIZE, desc=f"Shard {shard_idx}", unit="tok")

with mp.Pool(nprocs) as pool:
    for tokens in pool.imap(tokenize,data , chunksize = 16):
        needed = len(tokens)
    
        while needed > 0:
            space = SHARD_SIZE - token_count
            to_write = min(space , needed)
            offset = len(tokens) - needed

            current_tokens[token_count:token_count + to_write] = tokens[offset:offset + to_write]
            token_count += to_write
            needed -= to_write
            progress.update(to_write)

            if token_count == SHARD_SIZE:
                np.save(os.path.join(OUT_DIR, f"shard_{shard_idx:04d}.npy"), current_tokens)
                print(f"\nSaved shard {shard_idx}")
                shard_idx += 1
                token_count = 0
                progress = tqdm(total=SHARD_SIZE, desc=f"Shard {shard_idx}", unit="tok")

    
    if token_count > 0:
        np.save(os.path.join(OUT_DIR, f"shard_{shard_idx:04d}.npy"), current_tokens[:token_count])
        print(f"Saved final shard {shard_idx} ({token_count:,} tokens)")


    

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

sample/10BT/000_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/001_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/002_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

...

sample/10BT/011_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/012_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/013_00000.parquet:   0%|          | 0.00/541M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9672101 [00:00<?, ? examples/s]

Shard 0:   0%|          | 0/100000000 [00:00<?, ?tok/s]